# Lab M13 — Analysing and Forecasting Time Series (Starter)

> **Goal:** Process time-based data, build **baseline forecasts**, and evaluate them using **MAE** and **RMSE**.  
> **Dataset:** `validated_dataset_m13.csv`

---

## How to use this notebook
- Run cells top to bottom.
- Read the **markdown instructions** carefully.
- Fill in the **TODO** sections (you’ll see them clearly marked).
- Save outputs to the `output/` folder (created for you).

✅ By the end, you will have:
- time series plots  
- rolling averages  
- at least one baseline forecast  
- evaluation metrics (MAE, RMSE) saved to `output/forecast_metrics.csv`


In [ ]:
# 1) Setup
# If you run into missing packages, install them in your environment first.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Create output folders (safe to re-run)
Path("output/figures").mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")


## 2) Load and prepare the dataset

In time series work, **dates must be parsed correctly**.

You will:
1. Load the CSV from `data/validated_dataset_m13.csv`
2. Identify the **date column**
3. Convert it to `datetime`
4. Set it as the index


In [ ]:
from pathlib import Path
import pandas as pd

# Load data using a project-root anchored path (works when running from /notebooks)
PROJECT_ROOT = Path.cwd().parent
data_path = PROJECT_ROOT / "data" / "validated_dataset_m13.csv"

df = pd.read_csv(data_path)
df.head()


### Identify your date column

Below, inspect the columns and decide which column represents time.

✅ Common patterns: `date`, `order_date`, `created_at`, `timestamp`

**TODO:** Set `date_col` to the correct column name.


In [ ]:
df.columns

In [ ]:
# ---- Auto-detect date and target columns ----
candidate_date_cols = [
    c for c in df.columns
    if c.lower() in {"date", "ds", "timestamp", "order_date", "datetime"} or "date" in c.lower()
]

assert candidate_date_cols, f"No date-like column found. Columns: {list(df.columns)}"
date_col = candidate_date_cols[0]

numeric_cols = df.select_dtypes(include="number").columns.tolist()
assert numeric_cols, "No numeric columns available for time series analysis."
target_col = numeric_cols[0]

date_col, target_col


In [ ]:
# 3) Create a clean time series DataFrame
# - Keep only the date and target columns
# - Parse dates explicitly
# - Set the date as the index
# - Ensure chronological order

df_ts = df[[date_col, target_col]].copy()

# Parse the date column explicitly so we get a DatetimeIndex later
df_ts[date_col] = pd.to_datetime(df_ts[date_col], errors="coerce")

# Drop rows where the date or target could not be parsed
df_ts = df_ts.dropna(subset=[date_col, target_col])

df_ts = df_ts.set_index(date_col).sort_index()

# Sanity check: resample requires a DatetimeIndex
print("Index type:", type(df_ts.index))
df_ts.head()


In [ ]:
# Ensure the index is a DatetimeIndex before resampling
df_ts.index = pd.to_datetime(df_ts.index, errors="coerce")
df_ts = df_ts.dropna(subset=[target_col]).sort_index()

print(type(df_ts.index))


### Check for parsing issues

If some values could not be parsed, they become `NaT` (missing datetime).

**TODO:** Inspect missing datetime values. Decide how to handle them (drop or fix).


In [ ]:
# TODO: check how many dates failed parsing
df[date_col].isna().sum()


In [ ]:
# TODO: handle unparseable dates
# Option A: drop rows where the date couldn't be parsed
df = df.dropna(subset=[date_col]).copy()

# Set index for time series work
df = df.set_index(date_col).sort_index()

df.head()


## 3) Choose a numeric target to forecast

You need a **numeric time series** to forecast (e.g., daily revenue, daily orders).

**You will:**
1. Choose a numeric column to forecast (the "target")
2. Aggregate the dataset to a regular time frequency (daily or weekly)

**TODO:** Pick `target_col`.


In [ ]:
df.select_dtypes(include="number").columns

In [ ]:
# TODO: choose the numeric target column for forecasting
target_col = target_col  # already set in the load cell above

assert target_col is not None, "❗ Set target_col to the numeric column you want to forecast."


### Create a regular time series (daily)

Real-world datasets may have multiple rows per day.
We’ll aggregate to a **daily** series using `resample()`.

**TODO:** Decide on the aggregation method:
- **sum** for totals (e.g., revenue, number of orders)
- **mean** for averages (e.g., average order value)


In [ ]:
# ---- Resample the time series (requires a DatetimeIndex) ----
# Because the dataset has multiple records per day, we aggregate to a daily time series.
# For revenue, the most meaningful daily aggregation is a sum (total revenue per day).

import pandas as pd

# Ensure the index is a DatetimeIndex before resampling
df_ts.index = pd.to_datetime(df_ts.index, errors="coerce")
df_ts = df_ts.dropna(subset=[target_col]).sort_index()

agg_method = "sum"  # "sum" for daily totals, or "mean"

if agg_method == "sum":
    ts = df_ts[target_col].resample("D").sum()
elif agg_method == "mean":
    ts = df_ts[target_col].resample("D").mean()
else:
    raise ValueError("agg_method must be 'sum' or 'mean'")

ts.head()


## 4) Visualise the time series

A plot helps you notice:
- trends (up/down)
- seasonality (weekly patterns)
- noise/outliers


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(ts.index, ts.values)
plt.title(f"Time Series: {target_col} (daily)")
plt.xlabel("Date")
plt.ylabel(target_col)
plt.tight_layout()

plt.savefig("output/figures/time_series_raw.png", dpi=150)
plt.show()

print("✅ Saved: output/figures/time_series_raw.png")


## 5) Rolling averages (smoothing)

Rolling averages reduce noise and highlight trends.

**TODO:**
- Create rolling averages using windows like 7 (weekly) and 30 (monthly-ish)
- Plot them against the original series


In [ ]:
# TODO: try different windows
roll_7 = ts.rolling(window=7).mean()
roll_30 = ts.rolling(window=30).mean()

plt.figure(figsize=(12,4))
plt.plot(ts.index, ts.values, label="Daily")
plt.plot(roll_7.index, roll_7.values, label="7-day rolling avg")
plt.plot(roll_30.index, roll_30.values, label="30-day rolling avg")
plt.title("Rolling Averages")
plt.xlabel("Date")
plt.ylabel(target_col)
plt.legend()
plt.tight_layout()

plt.savefig("output/figures/rolling_averages.png", dpi=150)
plt.show()

print("✅ Saved: output/figures/rolling_averages.png")


## 6) Train/Test split

To evaluate forecasts, we split the series into:
- **train**: used to build baseline forecasts
- **test**: used to evaluate accuracy

We’ll keep the last ~20% as test data.


In [ ]:
test_size = 0.2
split_idx = int(len(ts) * (1 - test_size))

train = ts.iloc[:split_idx]
test = ts.iloc[split_idx:]

print(f"Train length: {len(train)}")
print(f"Test length:  {len(test)}")
print(f"Train end:    {train.index.max()}")
print(f"Test start:   {test.index.min()}")


## 7) Baseline forecasting models

Implement **simple baselines**:

1. **Naive forecast:** tomorrow = last observed value  
2. **Mean forecast:** predict the historical mean  
3. **Rolling-average forecast:** predict a recent rolling average  

Baselines set a **minimum standard** for performance.


In [ ]:
# 1) Naive forecast: predict the last training value for every test date
naive_forecast = pd.Series(train.iloc[-1], index=test.index)

# 2) Mean forecast: predict the mean of training values
mean_forecast = pd.Series(train.mean(), index=test.index)

# 3) Rolling average forecast: predict last 7-day rolling average from training
last_roll7 = train.rolling(window=7).mean().iloc[-1]
roll7_forecast = pd.Series(last_roll7, index=test.index)

naive_forecast.head(), mean_forecast.head(), roll7_forecast.head()


## 8) Evaluate forecasts with MAE and RMSE

### MAE (Mean Absolute Error)
Average absolute difference between predictions and actual values.

### RMSE (Root Mean Squared Error)
Squares errors first → penalises larger mistakes more.


In [ ]:
def mae(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

metrics = []

for name, pred in [
    ("naive", naive_forecast),
    ("mean", mean_forecast),
    ("rolling_7", roll7_forecast),
]:
    metrics.append({
        "model": name,
        "MAE": mae(test.values, pred.values),
        "RMSE": rmse(test.values, pred.values),
    })

metrics_df = pd.DataFrame(metrics).sort_values("RMSE")
metrics_df


In [ ]:
# Save metrics
metrics_df.to_csv("output/forecast_metrics.csv", index=False)
print("✅ Saved: output/forecast_metrics.csv")


## 9) Plot actual vs forecasts

A visual comparison often reveals *why* a model performs well or poorly.


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(train.index, train.values, label="Train (actual)")
plt.plot(test.index, test.values, label="Test (actual)")

plt.plot(test.index, naive_forecast.values, label="Naive forecast")
plt.plot(test.index, mean_forecast.values, label="Mean forecast")
plt.plot(test.index, roll7_forecast.values, label="Rolling-7 forecast")

plt.title("Actual vs Baseline Forecasts")
plt.xlabel("Date")
plt.ylabel(target_col)
plt.legend()
plt.tight_layout()

plt.savefig("output/figures/actual_vs_forecasts.png", dpi=150)
plt.show()

print("✅ Saved: output/figures/actual_vs_forecasts.png")


## 10) Interpret results (markdown)

**TODO:** In your own words, answer:

1. Which baseline performed best (lowest RMSE)?
2. Why might that baseline be stronger for this dataset?
3. What limitations exist in these forecasts?
4. What would you try next if you wanted a better model?


### Your Interpretation

- **Best baseline model:**  
  (write here)

- **Why it performed best:**  
  (write here)

- **Limitations / risks:**  
  (write here)

- **What you would try next:**  
  (write here)


## AI Reflection Prompt

Before finalising your submission, ask an AI assistant:

> **“Why are simple baseline forecasts important before applying advanced time series models?”**

Then reflect briefly:
- What did the answer clarify?
- What did you already do well in this lab?
- What would you change if the dataset updated weekly?


### Your AI Reflection

(write here)
